# Import important libraries

In [1]:
import numpy as np
from pathlib import Path
import torch
import glob
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models
from torchvision.models import ResNet18_Weights
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import skimage.io as skio
from skimage.feature import graycomatrix, graycoprops
import skimage.measure as skm
from skimage.filters import threshold_otsu 
from google.colab import drive
from tqdm import tqdm
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# unzip dataset

!unzip -q "/content/drive/MyDrive/BMET5933/AS2/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone_unique_medium.zip" -d "/content/drive/MyDrive/BMET5933/AS2"


In [15]:
# Hyperparameters configuration
image_size = 224
batch_size = 32
num_classes = 4
learning_rate = 1e-4
num_epochs = 30

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset class definition

In [16]:
# Dataset loading and preprocessing
class HandcraftedFeatures:
	'''
	input: PIL image
	output: list of handcrafted features
	'''
	# Convert the image to grayscale
	@staticmethod
	def get_gray_image(image):
		if image.mode != 'L':
			return np.array(image.convert('L'))
		return np.array(image)
	
	@staticmethod
	def quantize_8bit(gray_image):
		# Quantize the grayscale image to 8 levels
		quantized_image = (gray_image / 32).astype(np.uint8)
		return quantized_image
	
	# Otsu's thresholding to convert the image to binary
	@staticmethod
	def get_binary_mask(gray_image):
		thresh = threshold_otsu(gray_image)
		binary_mask = gray_image > thresh
		return binary_mask

	@staticmethod
	def extract_shape_features(image):
		# Example feature extraction using skimage
		gray_image = HandcraftedFeatures.get_gray_image(image)	
		# Otsu's thresholding to convert the image to binary
		binary_mask = HandcraftedFeatures.get_binary_mask(gray_image)
		# Label connected components in the binary mask
		labeled_mask = skm.label(binary_mask)
		regions = skm.regionprops(labeled_mask)
		# Extract features from the largest connected component
		largest_region = max(regions, key=lambda r: r.area)
		area = largest_region.area
		perimeter = largest_region.perimeter
		solidity = largest_region.solidity
		eccentricity = largest_region.eccentricity
		return [area, perimeter, solidity, eccentricity]
	
	@staticmethod
	def extract_texture_features(image):
		# Example feature extraction using skimage
		gray_image = HandcraftedFeatures.get_gray_image(image)
		# quantize the grayscale image to 8 levels
		quantized_image = HandcraftedFeatures.quantize_8bit(gray_image)
		# Otsu's thresholding to convert the image to binary
		binary_mask = HandcraftedFeatures.get_binary_mask(gray_image)
		# Label connected components in the binary mask
		labeled_mask = skm.label(binary_mask)
		regions = skm.regionprops(labeled_mask)
		# Extract features from the largest connected component
		largest_region = max(regions, key=lambda r: r.area)
		# Mask the quantized image to focus on the largest region
		masked_image = quantized_image * (labeled_mask == largest_region.label)
		texture_features = []
		angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
		glcm_angle = graycomatrix(np.array(masked_image), distances=[1], angles=angles, levels=8, symmetric=True, normed=True)
		for i, _ in enumerate(angles):
			contrast = graycoprops(glcm_angle, 'contrast')[0, i]
			dissimilarity = graycoprops(glcm_angle, 'dissimilarity')[0, i]
			homogeneity = graycoprops(glcm_angle, 'homogeneity')[0, i]
			texture_features.extend([contrast, dissimilarity, homogeneity])
		return texture_features
	
	@staticmethod
	def extract_color_features(image):
		# feature extraction using skimage
		rgb_image = np.array(image)
		mean_red = np.mean(rgb_image[:, :, 0])
		mean_green = np.mean(rgb_image[:, :, 1])
		mean_blue = np.mean(rgb_image[:, :, 2])
		return [mean_red, mean_green, mean_blue]
	
	# Combine all features into a single feature vector
	@staticmethod
	def extract_all(image):
		shape_features = HandcraftedFeatures.extract_shape_features(image)
		texture_features = HandcraftedFeatures.extract_texture_features(image)
		color_features = HandcraftedFeatures.extract_color_features(image)
		return shape_features + texture_features + color_features


In [ ]:
# all feature columns
feature_columns = [
	"area", "perimeter", "solidity", "eccentricity",

	"contrast_0", "dissimilarity_0", "homogeneity_0",
	"contrast_45", "dissimilarity_45", "homogeneity_45",
	"contrast_90", "dissimilarity_90", "homogeneity_90",
	"contrast_135", "dissimilarity_135", "homogeneity_135",

	"mean_red", "mean_green", "mean_blue"
]

def get_image_paths(dataset_root):
	'''
		Get all image paths by root_dir
		Input:
			root_dir: directory containing images (can have subdirectories for classes)
		Output:
			sorted list of image paths
	'''
	root_dir = Path(root_dir)
	image_paths = []
	image_paths.extend(root_dir.rglob("*.jpg"))
	return sorted(list(image_paths))

def extract_features_to_csv(path_list, dataset_indices, output_csv):
	'''
		Extract handcrafted features from images in dataset_root and save to output_csv
		Input:
			dataset_root: directory containing images (can have subdirectories for classes)
			output_csv: path to save the extracted features in csv format
	'''
	image_paths = [path_list[i] for i in dataset_indices]

	rows = []
	for image_path in tqdm(image_paths, desc = f"Extracting features from dataset"):

		# create dict row, storing path and label
		row = {
			'path': str(image_path),
			'label': image_path.parent.name
			}
		# read image and extract features
		image = Image.open(image_path)
		features = HandcraftedFeatures.extract_all(image)

		# add handcrafted features to dict row
		row.update({col: feature for col, feature in zip(feature_columns, features)})
		rows.append(row)

	# store features in csv file
	df = pd.DataFrame(rows)
	df.to_csv(output_csv, index=False)



In [ ]:
# dataset paths
dataset_root = Path("/content/drive/MyDrive/BMET5933/AS2/train")

image_paths = get_image_paths(dataset_root)
img_indices = np.arange(len(image_paths))

# split dataset into train, val and test sets with stratification
train_indices, temp_indices = train_test_split(img_indices, 
											   test_size=0.2, 
											   random_state=42, 
											   stratify=[path.parent.name for path in image_paths])

val_indices, test_indices = train_test_split(temp_indices,
											 test_size=0.5, 
											 random_state=42, 
											 stratify=[path.parent.name for path in np.array(image_paths)[temp_indices]])

# Extract features for train, val, and test sets
train_csv = Path("/content/drive/MyDrive/BMET5933/AS2/train_features.csv")
val_csv = Path("/content/drive/MyDrive/BMET5933/AS2/val_features.csv")
test_csv = Path("/content/drive/MyDrive/BMET5933/AS2/test_features.csv")

# Encode labels first
label_encoder = LabelEncoder()
labels = [class_dir.name for class_dir in dataset_root.glob('*') if class_dir.is_dir()]
label_encoder.fit(labels)

# extract features if the csv files do not already exist
if not train_csv.exists() or not val_csv.exists() or not test_csv.exists():
	extract_features_to_csv(image_paths, train_indices, train_csv)
	extract_features_to_csv(image_paths, val_indices, val_csv)
	extract_features_to_csv(image_paths, test_indices, test_csv)

In [ ]:
# Define the dataset class
class Kidney_Dataset(Dataset):
	def __init__(self, dataset_root, handcrafted_features_csv, transform=None, label_encoder=None):
		super().__init__()
		self.dataset_root = dataset_root
		self.transform = transform
		self.handcrafted_features = pd.read_csv(handcrafted_features_csv)
		self.image_paths = self.handcrafted_features['path'].tolist()
		self.labels = [Path(path).parent.name for path in self.image_paths]
		
		self.label_encoder = label_encoder
	
	def encode_label(self, label):
		# label encoder need and output list
		return self.label_encoder.transform([label])[0]
	
	def decode_label(self, encoded_label):
		return self.label_encoder.inverse_transform([encoded_label])[0]
	
	def __len__(self):
		return len(self.image_paths)
	
	def __getitem__(self, idx):
		image_path = self.image_paths[idx]
		label = self.labels[idx]
		
		original_image = Image.open(image_path).convert("RGB")

		# Extract handcrafted features
		handcrafted_features = torch.tensor(self.handcrafted_features.iloc[idx, 2:].values, dtype=torch.float32)

		if self.transform:
			transformed_image = self.transform(original_image)
		
		encoded_label = self.encode_label(label)
		return transformed_image, handcrafted_features, encoded_label
	
	
# Define transformations for the dataset
transform = transforms.Compose(
	[transforms.Resize((image_size, image_size)),
	 transforms.ToTensor(),
	 transforms.Normalize(
		 mean = [0.485, 0.456, 0.406],
		 std = [0.229, 0.224, 0.225]
	 )]
)


# Dataset initialization

In [20]:
# Create dataset instances for train, validation and test sets
train_dataset = Kidney_Dataset(train_root, train_csv, transform=transform, label_encoder=label_encoder)
validation_dataset = Kidney_Dataset(validation_root, val_csv, transform=transform, label_encoder=label_encoder)
test_dataset = Kidney_Dataset(test_root, test_csv, transform=transform, label_encoder=label_encoder)

# Dataloaders for training, validation and testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model custom modification

In [21]:
class DL_model(nn.Module):
	def __init__(self):
		super().__init__()
		self.resnet = models.resnet18(weights = ResNet18_Weights.DEFAULT)
		self.feature_dim = self.resnet.fc.in_features
		self.resnet.fc = nn.Identity()  
	
	def forward(self, x):
		x = self.resnet(x)
		return x


class Fusion_model(nn.Module):
	'''
	fuse features from Resnet and handcrafted features
	'''
	def __init__(self, num_classes):
		super().__init__()
		self.resnet = DL_model()
		self.resnet_num_features = self.resnet.feature_dim
		self.handcrafted_num_features = 19
		
		# a simle mlp for fusion classification
		self.mlp = nn.Sequential(
			nn.Linear(self.resnet_num_features + self.handcrafted_num_features, 256),
			nn.ReLU(),
			nn.Dropout(0.3),
			nn.Linear(256, num_classes)
		)
	
	def forward(self, image, handcrafted_features):
		cnn_features = self.resnet(image)
		combined_features = torch.cat((cnn_features, handcrafted_features), dim=1)
		output = self.mlp(combined_features)
		return output

In [22]:
# Initialize the model
model = Fusion_model(num_classes).to(device)
print(model)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

Fusion_model(
  (resnet): DL_model(
    (resnet): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): Batch

In [23]:
def train_loop(model, train_loader, criterion, optimizer, lr_scheduler, device, num_epochs	):
	# Training loop
	for epoch in range(num_epochs):
		model.train()
		train_loss = 0.0
		correct = 0
		total = 0
		tqdm_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
		for images, handcrafted_features, labels in tqdm_bar:
			images = images.to(device)
			handcrafted_features = handcrafted_features.to(device)
			labels = labels.to(device)
			
			# Forward pass and backward pass
			optimizer.zero_grad()
			outputs = model(images, handcrafted_features)
			loss = criterion(outputs, labels)
			loss.backward()
			optimizer.step()
			
			# Accumulate loss and calculate accuracy
			train_loss += loss.item() * images.size(0)
			predicted = torch.argmax(outputs, dim=1)
			correct += (predicted == labels).sum().item()
			total += labels.size(0)
		
		lr_scheduler.step()
		
		avg_train_loss = train_loss / len(train_loader.dataset)
		train_accuracy = correct / total
		print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.4f}")



# Evaluate the model on the validation set
def evaluate(model, validation_loader, criterion, device):
	model.eval()
	validation_loss = 0.0
	correct = 0
	total = 0
	# Do not calculate gradients during evaluation
	with torch.no_grad():
		for images, handcrafted_features, labels in validation_loader:
			images = images.to(device)
			handcrafted_features = handcrafted_features.to(device)
			labels = labels.to(device)
			
			outputs = model(images, handcrafted_features)
			loss = criterion(outputs, labels)
			
			validation_loss += loss.item() * images.size(0)
			predicted = torch.argmax(outputs, dim=1)
			correct += (predicted == labels).sum().item()
			total += labels.size(0)
	
	avg_validation_loss = validation_loss / len(validation_loader.dataset)
	validation_accuracy = correct / total
	print(f"Validation Loss: {avg_validation_loss:.4f}, Accuracy: {validation_accuracy:.4f}")
	return avg_validation_loss, validation_accuracy



# Test the model and print classification report and confusion matrix
def test_model(model, test_loader, device, label_encoder):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, handcrafted_features, labels in test_loader:
            images = images.to(device)
            handcrafted_features = handcrafted_features.to(device)
            labels = labels.to(device)
            outputs = model(images, handcrafted_features)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    target_names = [str(class_name) for class_name in label_encoder.classes_]

    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=target_names))

    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

In [24]:
# Train the model
train_loop(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    device=device,
    num_epochs=num_epochs
)

# Evaluate the model on the validation set
evaluate(
    model=model,
    validation_loader=validation_loader,
    criterion=criterion,
    device=device
)



Epoch 1/30:   0%|          | 0/131 [00:00<?, ?it/s]


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.